In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Installation


In [ ]:
!pip install rouge-score pycocoevalcap

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 9.3 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=716206156afbdd9e2195867d450ea3001de0d616fdee053289dbe5cae595a3ab
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


# Lib


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import re
import ast
import math
import glob
import copy
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import kagglehub
from PIL import Image
from tqdm import tqdm
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import CosineAnnealingLR

from transformers import AutoTokenizer, AutoModel

import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from pycocoevalcap.cider.cider import Cider
from rouge_score import rouge_scorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

# Data Preparation


## Utils

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def print_directory_structure(startpath, max_level=3):
    start_level = startpath.count(os.sep)

    for root, dirs, files in os.walk(startpath):
        level = root.count(os.sep) - start_level
        if level > max_level:
            del dirs[:]
            continue

        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')

        if level < max_level:
            subindent = ' ' * 4 * (level + 1)
            for f in files:
                print(f'{subindent}{f}')

def parse_list_string(val):
    if pd.isna(val):
        return []
    try:
        parsed = ast.literal_eval(val)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

def select_best_single_image(row):
    pa_list = parse_list_string(row["PA"])
    if pa_list:
        return pa_list[0]

    ap_list = parse_list_string(row["AP"])
    if ap_list:
        return ap_list[0]

    img_list = parse_list_string(row["image"])
    if img_list:
        return img_list[0]

    return None

def extract_all_images_from_row(row):
    all_imgs = []

    pa_list = parse_list_string(row["PA"])
    if pa_list:
        all_imgs.extend(pa_list)

    ap_list = parse_list_string(row["AP"])
    if ap_list:
        all_imgs.extend(ap_list)

    img_list = parse_list_string(row["image"])
    if img_list:
        all_imgs.extend(img_list)

    return list(set(all_imgs))

def filter_existing_images(df, root_dir, image_col='best_image'):
    exists_mask = []
    for img_path in tqdm(df[image_col]):
        if pd.isna(img_path):
            exists_mask.append(False)
            continue
        full_path = os.path.join(root_dir, img_path)
        exists_mask.append(os.path.exists(full_path))
    filtered_df = df[exists_mask].reset_index(drop=True)
    return filtered_df

def clean_stringified_list(val):
    if pd.isna(val):
        return val
    try:
        parsed_list = ast.literal_eval(val)
        if isinstance(parsed_list, list) and len(parsed_list) > 0:
            return parsed_list[0]
        return val
    except (ValueError, SyntaxError):
        return val

## Dowload

In [ ]:
path = kagglehub.dataset_download("simhadrisadaram/mimic-cxr-dataset")
DATA_DIR_ROOT = path
print("Path to dataset files:", path)

100%|██████████| 16.5G/16.5G [03:31<00:00, 83.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/simhadrisadaram/mimic-cxr-dataset/versions/2


In [ ]:
#print_directory_structure(path)

In [ ]:
DATA_DIR = path
ROOT_DIR = os.path.join(DATA_DIR_ROOT, "official_data_iccv_final")
TRAIN_CSV = os.path.join(DATA_DIR, "mimic_cxr_aug_train.csv")
VAL_CSV = os.path.join(DATA_DIR, "mimic_cxr_aug_validate.csv")

print(f"DATA_DIR: {DATA_DIR}")
print(f"TRAIN CSV: {TRAIN_CSV}")
print(f"VAL CSV: {VAL_CSV}")

DATA_DIR: /root/.cache/kagglehub/datasets/simhadrisadaram/mimic-cxr-dataset/versions/2
TRAIN CSV: /root/.cache/kagglehub/datasets/simhadrisadaram/mimic-cxr-dataset/versions/2/mimic_cxr_aug_train.csv
VAL CSV: /root/.cache/kagglehub/datasets/simhadrisadaram/mimic-cxr-dataset/versions/2/mimic_cxr_aug_validate.csv


## Splitting

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
df_raw = pd.concat([train_df, val_df], ignore_index=True)

In [ ]:
df_raw = df_raw.sample(frac=1, random_state=42).reset_index(drop=True)

final_records = []
seen_images = set()
target_size = 74000

for row in df_raw.to_dict("records"):
    img_list = parse_list_string(row.get("image", ""))
    raw_view = row.get("view", "")
    raw_text = row.get("text", "")
    parsed_text = parse_list_string(raw_text)

    findings_text = ""
    for section in parsed_text:
        if "findings" in section.lower():
            findings_text = section
            break

    if not findings_text or len(findings_text) < 30:
        continue

    for img in img_list:
        if img in seen_images:
            continue

        img_path = os.path.join(ROOT_DIR, img)
        if os.path.exists(img_path):
            seen_images.add(img)

            sub_id = row.get(
                "subject_id",
                row.get("Subject_id", row.get("SUBJECT_ID", None)),
            )

            final_records.append(
                {
                    "subject_id": sub_id,
                    "view": raw_view,
                    "best_image": img,
                    "path": img,
                    "text": findings_text,
                }
            )

        if len(seen_images) >= target_size:
            break

    if len(seen_images) >= target_size:
        break

df_all = pd.DataFrame(final_records)
print(f"Total images: {len(df_all)}")

Total images: 74000


In [ ]:
invalid_text_df = df_all[
    (df_all['text'].str.len() < 31) ]

if invalid_text_df.empty:
    print("No records found with text length < 30 or containing 'impression'.")
else:
    print(f"Found {len(invalid_text_df)} invalid records:")
    display(invalid_text_df)

Found 10 invalid records:


,subject_id,view,best_image,path,text
6503,13461141,"['PA', 'LL']",files/p13/p13461141/s55563575/1724cc70-7794dca...,files/p13/p13461141/s55563575/1724cc70-7794dca...,Findings: Impression: Normal.
6504,13461141,"['PA', 'LL']",files/p13/p13461141/s55563575/a23f8bd1-aa6c312...,files/p13/p13461141/s55563575/a23f8bd1-aa6c312...,Findings: Impression: Normal.
31530,16551790,"['PA', 'LL', 'LATERAL']",files/p16/p16551790/s51993504/836098e0-074451e...,files/p16/p16551790/s51993504/836098e0-074451e...,Findings: Impression: Normal.
31531,16551790,"['PA', 'LL', 'LATERAL']",files/p16/p16551790/s51993504/e3629026-04b23e9...,files/p16/p16551790/s51993504/e3629026-04b23e9...,Findings: Impression: Normal.
31532,16551790,"['PA', 'LL', 'LATERAL']",files/p16/p16551790/s52546529/294c240e-9ea3ab8...,files/p16/p16551790/s52546529/294c240e-9ea3ab8...,Findings: Impression: Normal.
31533,16551790,"['PA', 'LL', 'LATERAL']",files/p16/p16551790/s52546529/4190c369-d2017c0...,files/p16/p16551790/s52546529/4190c369-d2017c0...,Findings: Impression: Normal.
31534,16551790,"['PA', 'LL', 'LATERAL']",files/p16/p16551790/s52546529/8e9bf331-0dd7d81...,files/p16/p16551790/s52546529/8e9bf331-0dd7d81...,Findings: Impression: Normal.
31535,16551790,"['PA', 'LL', 'LATERAL']",files/p16/p16551790/s56374011/4abc3a8e-933ab0f...,files/p16/p16551790/s56374011/4abc3a8e-933ab0f...,Findings: Impression: Normal.
31536,16551790,"['PA', 'LL', 'LATERAL']",files/p16/p16551790/s56374011/4ec75506-373ef92...,files/p16/p16551790/s56374011/4ec75506-373ef92...,Findings: Impression: Normal.
31537,16551790,"['PA', 'LL', 'LATERAL']",files/p16/p16551790/s56374011/ae3fee57-51052c8...,files/p16/p16551790/s56374011/ae3fee57-51052c8...,Findings: Impression: Normal.


In [ ]:
grouped_patients = [
    patient_df
    for _, patient_df in df_all.groupby("subject_id")
]

rng = np.random.default_rng(42)
rng.shuffle(grouped_patients)


TRAIN_TARGET = 50000
VAL_TARGET = 10000
TEST_TARGET = 10000

train_chunks = []
val_chunks = []
test_chunks = []

train_count = 0
val_count = 0
test_count = 0


for patient_df in grouped_patients:

    n_imgs = len(patient_df)

    if train_count < TRAIN_TARGET:
        train_chunks.append(patient_df)
        train_count += n_imgs

    elif val_count < VAL_TARGET:
        val_chunks.append(patient_df)
        val_count += n_imgs

    elif test_count < TEST_TARGET:
        test_chunks.append(patient_df)
        test_count += n_imgs

    else:
        break


train_df_final = pd.concat(train_chunks, ignore_index=True)
val_df_final = pd.concat(val_chunks, ignore_index=True)
test_df_final = pd.concat(test_chunks, ignore_index=True)

train_subjects = set(train_df_final["subject_id"])
val_subjects = set(val_df_final["subject_id"])
test_subjects = set(test_df_final["subject_id"])

print("Train ∩ Val :", len(train_subjects & val_subjects))
print("Train ∩ Test:", len(train_subjects & test_subjects))
print("Val ∩ Test  :", len(val_subjects & test_subjects))
print()
print(
    f"Train : {len(train_df_final):,} images | "
    f"{len(train_subjects):,} patients"
)

print(
    f"Val   : {len(val_df_final):,} images | "
    f"{len(val_subjects):,} patients"
)

print(
    f"Test  : {len(test_df_final):,} images | "
    f"{len(test_subjects):,} patients"
)

Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0

Train : 50,009 images | 8,863 patients
Val   : 10,000 images | 1,819 patients
Test  : 10,003 images | 1,791 patients


In [ ]:
print(f"Unique texts in Train: {train_df_final['text'].nunique()}")
print(f"Unique texts in Val: {val_df_final['text'].nunique()}")
print(f"Unique texts in Test: {test_df_final['text'].nunique()}")

Unique texts in Train: 8061
Unique texts in Val: 1714
Unique texts in Test: 1701


In [ ]:
train_ids = set(train_df_final["subject_id"])
val_ids = set(val_df_final["subject_id"])
test_ids = set(test_df_final["subject_id"])

print(len(train_ids & val_ids))
print(len(train_ids & test_ids))
print(len(val_ids & test_ids))

0
0
0


In [ ]:
train_df_final.to_csv("/content/drive/MyDrive/UEH TERMS/Kì 6/Tutorial Deep Learning/colab/data/mimic_train.csv", index=False)
val_df_final.to_csv("/content/drive/MyDrive/UEH TERMS/Kì 6/Tutorial Deep Learning/colab/data/mimic_val.csv", index=False)
test_df_final.to_csv("/content/drive/MyDrive/UEH TERMS/Kì 6/Tutorial Deep Learning/colab/data/mimic_test.csv", index=False)

In [ ]:
train_df_final.head()

,subject_id,view,best_image,path,text
0,15761456,['AP'],files/p15/p15761456/s54426882/e2c2ff53-f618217...,files/p15/p15761456/s54426882/e2c2ff53-f618217...,"Findings: Compared to ___, the lung volumes ar..."
1,15761456,['AP'],files/p15/p15761456/s54662596/0081fa27-d27d9e0...,files/p15/p15761456/s54662596/0081fa27-d27d9e0...,"Findings: Compared to ___, the lung volumes ar..."
2,15761456,['AP'],files/p15/p15761456/s54667432/dc363540-f4cadd1...,files/p15/p15761456/s54667432/dc363540-f4cadd1...,"Findings: Compared to ___, the lung volumes ar..."
3,15761456,['AP'],files/p15/p15761456/s56791567/8df7c24d-3c5a7c3...,files/p15/p15761456/s56791567/8df7c24d-3c5a7c3...,"Findings: Compared to ___, the lung volumes ar..."
4,10602633,"['nan', 'PA', 'LATERAL', 'AP']",files/p10/p10602633/s51004997/cb1e7ca3-159b607...,files/p10/p10602633/s51004997/cb1e7ca3-159b607...,Findings: As compared to the previous radiogra...


In [ ]:
train_df_final.iloc[1, -1]

'Findings: Compared to ___, the lung volumes are lower, likely due to increased bilateral atelectasis.  Moderate right and small left pleural effusion persists.  Round atelectasis on the right is partially visualized Pulmonary edema has nearly resolved.  The heart size is difficult to determine, though unlikely unchanged.  Pleural plaques are unchanged.  Right clavicle, scapular fracture appear unchanged.  Vertebral fixation with screws and rods are well aligned.  No pneumothorax is seen. Impression: Increased bilateral atelectasis and pleural effusion, right worse than left.'